# Notebook 05: Inventory Optimization
## Retail Demand Forecasting & Inventory Optimization

**Sections:** ABC Classification | EOQ | Safety Stock | Reorder Point | Stockout Risk | Financial Impact

**Inputs:** `Data/processed/` parquet files only - no database required.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

DATA_PROC = Path('../Data/processed')
IMAGES    = Path('../images')
IMAGES.mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': '#0f172a', 'axes.facecolor': '#1e293b',
    'axes.edgecolor': '#334155', 'text.color': '#f1f5f9',
    'axes.labelcolor': '#f1f5f9', 'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8', 'grid.color': '#334155',
})
PALETTE = ['#38bdf8', '#34d399', '#fb923c', '#818cf8', '#f472b6']

print('Inventory optimization environment ready')

In [ ]:
df = pd.read_parquet(DATA_PROC / 'sales_enriched_sample.parquet')
df['date'] = pd.to_datetime(df['date'])

print(f'Sample: {len(df):,} rows | {df.product_id.nunique()} SKUs | {df.date.min().date()} to {df.date.max().date()}')

## Phase A: ABC Classification

In [ ]:
abc_data = (
    df.assign(revenue=lambda x: x['units_sold'] * x['sell_price'].fillna(0))
    .groupby(['product_id', 'store_id', 'cat_id'])
    .agg(total_units=('units_sold', 'sum'), total_revenue=('revenue', 'sum'))
    .reset_index()
    .sort_values('total_revenue', ascending=False)
)

total_rev = abc_data['total_revenue'].sum()
abc_data['revenue_pct']    = abc_data['total_revenue'] / total_rev
abc_data['cumulative_pct'] = abc_data['revenue_pct'].cumsum()
abc_data['abc_class'] = abc_data['cumulative_pct'].apply(
    lambda x: 'A' if x <= 0.70 else ('B' if x <= 0.90 else 'C')
)

abc_summary = abc_data.groupby('abc_class').agg(
    num_skus=('product_id', 'count'),
    total_revenue=('total_revenue', 'sum')
).reset_index()
abc_summary['sku_pct'] = (abc_summary['num_skus'] / abc_summary['num_skus'].sum() * 100).round(1)
abc_summary['rev_pct'] = (abc_summary['total_revenue'] / abc_summary['total_revenue'].sum() * 100).round(1)

print('ABC Classification Results:')
display(abc_summary)

In [ ]:
abc_colors = {'A': '#34d399', 'B': '#38bdf8', 'C': '#fb923c'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ABC Inventory Classification', fontsize=14, color='white')

# Pareto curve
bar_colors = [abc_colors[c] for c in abc_data['abc_class']]
axes[0].bar(range(len(abc_data)), abc_data['cumulative_pct'] * 100,
            color=bar_colors, alpha=0.7, width=1.0)
axes[0].axhline(70, color='white', linestyle='--', linewidth=1.2, label='70% (A cutoff)')
axes[0].axhline(90, color='#94a3b8', linestyle='--', linewidth=1.2, label='90% (B cutoff)')
axes[0].set_title('Pareto Curve', color='white')
axes[0].set_xlabel('SKUs (sorted by revenue)')
axes[0].set_ylabel('Cumulative Revenue %')
axes[0].legend()

axes[1].pie(
    abc_summary['total_revenue'],
    labels=[f"Class {r['abc_class']} ({r['rev_pct']}% rev, {r['sku_pct']}% SKUs)"
            for _, r in abc_summary.iterrows()],
    colors=[abc_colors[c] for c in abc_summary['abc_class']],
    autopct='%1.1f%%', startangle=90,
    textprops={'color': 'white', 'fontsize': 9},
    wedgeprops={'edgecolor': '#0f172a', 'linewidth': 2}
)
axes[1].set_title('Revenue Share by ABC Class', color='white')

plt.tight_layout()
plt.savefig(IMAGES / '05_abc_classification.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

# Save to parquet
abc_data.to_parquet(DATA_PROC / 'abc_classification.parquet', index=False)
print(f'Saved abc_classification.parquet: {len(abc_data):,} rows')

## Phase B: EOQ, Safety Stock, Reorder Point

In [ ]:
# Parameters
LEAD_TIME    = 7      # days
SERVICE_LVL  = 0.95   # 95% service level
Z_SCORE      = stats.norm.ppf(SERVICE_LVL)
ORDER_COST   = 50     # $ per order
HOLDING_RATE = 0.25   # 25% of unit value per year

print(f'Lead Time: {LEAD_TIME}d | Service Level: {SERVICE_LVL*100:.0f}% | Z={Z_SCORE:.3f}')

# Per-SKU demand statistics
sku_stats = (
    df.groupby(['product_id', 'store_id', 'cat_id'])
    .agg(
        avg_daily_demand=('units_sold', 'mean'),
        demand_std=('units_sold', 'std'),
        avg_price=('sell_price', 'mean'),
        total_units=('units_sold', 'sum')
    )
    .reset_index()
)
sku_stats['demand_std'] = sku_stats['demand_std'].fillna(0)
sku_stats['avg_price']  = sku_stats['avg_price'].fillna(1.0)

# Formulas
sku_stats['safety_stock']  = (Z_SCORE * sku_stats['demand_std'] * np.sqrt(LEAD_TIME)).clip(0).round(0).astype(int)
sku_stats['reorder_point'] = (sku_stats['avg_daily_demand'] * LEAD_TIME + sku_stats['safety_stock']).round(0).astype(int)

annual_demand = sku_stats['avg_daily_demand'] * 365
holding_cost  = sku_stats['avg_price'] * HOLDING_RATE
sku_stats['eoq'] = np.sqrt(2 * annual_demand * ORDER_COST / holding_cost.replace(0, 0.01)).round(0).astype(int)

print(f'\nComputed for {len(sku_stats):,} SKU-Store combinations')
display(sku_stats.nlargest(8, 'avg_daily_demand')[
    ['product_id','store_id','cat_id','avg_daily_demand','demand_std','safety_stock','reorder_point','eoq']
].round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Inventory Metrics Distribution by Category', fontsize=13, color='white')

for ax, (metric, label, color) in zip(axes, [
    ('safety_stock', 'Safety Stock (Units)', '#38bdf8'),
    ('reorder_point', 'Reorder Point (Units)', '#34d399'),
    ('eoq', 'EOQ - Units per Order', '#fb923c'),
]):
    cap = sku_stats[metric].quantile(0.95)
    for cat, clr in [('FOODS','#34d399'),('HOUSEHOLD','#38bdf8'),('HOBBIES','#fb923c')]:
        ax.hist(sku_stats[(sku_stats['cat_id']==cat) & (sku_stats[metric]<cap)][metric],
                bins=30, alpha=0.6, label=cat, color=clr)
    ax.set_title(label, color='white')
    ax.set_xlabel('Units')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(IMAGES / '05_inventory_metrics.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

## Phase C: Stockout Risk Detection

In [ ]:
# Use last 14 days sales as a stock proxy
last_14 = df[df['date'] >= df['date'].max() - pd.Timedelta(days=14)]
stock_proxy = (
    last_14.groupby(['product_id', 'store_id'])['units_sold'].sum()
    .reset_index().rename(columns={'units_sold': 'current_stock'})
)

stockout = sku_stats.merge(stock_proxy, on=['product_id','store_id'], how='left')
stockout['current_stock'] = stockout['current_stock'].fillna(0).astype(int)
stockout['forecast_7d']   = (stockout['avg_daily_demand'] * 7).round(1)
stockout['risk_score']    = stockout['forecast_7d'] / stockout['current_stock'].replace(0, 0.1)
stockout['days_to_stockout'] = (stockout['current_stock'] / stockout['avg_daily_demand'].replace(0, 0.01)).round(0).astype(int).clip(0, 999)

def classify_risk(s):
    if s >= 1.5: return 'HIGH'
    elif s >= 0.8: return 'MEDIUM'
    return 'LOW'

stockout['risk_level'] = stockout['risk_score'].apply(classify_risk)

print('Stockout Risk Summary:')
display(stockout['risk_level'].value_counts().reset_index(name='count'))

print('\nTop HIGH Risk Products:')
display(stockout[stockout['risk_level']=='HIGH']
        .nlargest(10, 'risk_score')
        [['product_id','store_id','cat_id','current_stock','forecast_7d','risk_score','days_to_stockout']].round(2))

In [ ]:
risk_colors = {'HIGH': '#f87171', 'MEDIUM': '#fbbf24', 'LOW': '#34d399'}
risk_counts = stockout['risk_level'].value_counts().reset_index()
risk_counts.columns = ['risk_level', 'count']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Stockout Risk Analysis', fontsize=13, color='white')

axes[0].bar(risk_counts['risk_level'], risk_counts['count'],
            color=[risk_colors[r] for r in risk_counts['risk_level']],
            edgecolor='#0f172a', alpha=0.9)
axes[0].set_title('SKUs by Risk Level', color='white')
axes[0].set_ylabel('SKU-Store Count')
for i, (_, row) in enumerate(risk_counts.iterrows()):
    axes[0].text(i, row['count'] + 0.5, str(row['count']), ha='center', color='white', fontsize=11, fontweight='bold')

axes[1].pie(risk_counts['count'],
            labels=risk_counts['risk_level'],
            colors=[risk_colors[r] for r in risk_counts['risk_level']],
            autopct='%1.1f%%', startangle=90,
            textprops={'color':'white'},
            wedgeprops={'edgecolor':'#0f172a','linewidth':2})
axes[1].set_title('Risk Distribution', color='white')

plt.tight_layout()
plt.savefig(IMAGES / '05_stockout_risk.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

## Phase D: Financial Impact

In [ ]:
GROSS_MARGIN = 0.30

stockout['overstock']         = (stockout['current_stock'] - stockout['forecast_7d']).clip(lower=0)
stockout['understock']        = (stockout['forecast_7d'] - stockout['current_stock']).clip(lower=0)
stockout['holding_cost_30d']  = (stockout['overstock'] * stockout['avg_price'] * HOLDING_RATE / 12).round(2)
stockout['lost_revenue']      = (stockout['understock'] * stockout['avg_price']).round(2)
stockout['lost_profit']       = (stockout['lost_revenue'] * GROSS_MARGIN).round(2)

fin = (
    stockout.groupby('cat_id')
    .agg(holding_cost=('holding_cost_30d','sum'), lost_profit=('lost_profit','sum'),
         high_risk=('risk_level', lambda x: (x=='HIGH').sum()))
    .reset_index()
)
fin['total_impact'] = fin['holding_cost'] + fin['lost_profit']

total_impact = fin['total_impact'].sum()
print(f'Monthly Financial Impact by Category:')
display(fin.round(2))
print(f'\nTotal Estimated Monthly Impact: ${total_impact:,.0f}')

## Save All Inventory Outputs

In [ ]:
# Inventory recommendations
inv_rec = sku_stats.copy()
inv_rec['forecast_7d']           = (inv_rec['avg_daily_demand'] * 7).round(1)
inv_rec['forecast_30d']          = (inv_rec['avg_daily_demand'] * 30).round(1)
inv_rec['lead_time_days']        = LEAD_TIME
inv_rec['current_stock']         = stock_proxy.set_index(['product_id','store_id'])['current_stock'].reindex(
    pd.MultiIndex.from_frame(inv_rec[['product_id','store_id']])
).values
inv_rec['current_stock']         = inv_rec['current_stock'].fillna(0).astype(int)
inv_rec['recommended_order_qty'] = np.where(
    inv_rec['current_stock'] < inv_rec['reorder_point'], inv_rec['eoq'], 0
)

inv_rec.to_parquet(DATA_PROC / 'inventory_recommendations.parquet', index=False)
stockout[['product_id','store_id','cat_id','current_stock','forecast_7d',
           'risk_score','risk_level','days_to_stockout']].to_parquet(
    DATA_PROC / 'stockout_risk.parquet', index=False)
abc_data.to_parquet(DATA_PROC / 'abc_classification.parquet', index=False)

print('Saved parquet files:')
for f in ['inventory_recommendations.parquet', 'stockout_risk.parquet', 'abc_classification.parquet']:
    size = (DATA_PROC / f).stat().st_size / 1e3
    print(f'  {f}  ({size:.0f} KB)')

print('\n=== EXECUTIVE SUMMARY ===')
print(f'  Class A SKUs (70% revenue): {(abc_data.abc_class=="A").sum()}')
print(f'  Class B SKUs (20% revenue): {(abc_data.abc_class=="B").sum()}')
print(f'  Class C SKUs (10% revenue): {(abc_data.abc_class=="C").sum()}')
print(f'  HIGH stockout risk         : {(stockout.risk_level=="HIGH").sum()} SKU-Store combinations')
print(f'  SKUs to reorder NOW        : {(inv_rec.recommended_order_qty > 0).sum()}')
print(f'  Est. monthly financial impact: ${total_impact:,.0f}')
print('\nAll outputs saved. Proceed to dashboard/app.py')